In [ ]:
import os
import os

# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = "1"
# os.environ["TF_NUM_THREADS"] = "1"

if "CUDA_VISIBLE_DEVICES" not in os.environ:
    os.environ["CUDA_VISIBLE_DEVICES"] = "2"


import tensorflow as tf

gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
        # tf.config.experimental.set_virtual_device_configuration(
        #     gpu, [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=2500)]
        # )
    logical_gpus = tf.config.experimental.list_logical_devices("GPU")
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")


from pathlib import Path
import numpy as np
import pandas as pd

import os
import numpy as np
import random
from optuna.samplers import CmaEsSampler
from tensorflow.keras.activations import selu, gelu

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from tensorflow.keras.activations import selu, gelu
from optuna.samplers import CmaEsSampler
import os
import numpy as np
import random

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import (
    MinMaxScaler,
    QuantileTransformer,
    PowerTransformer,
    MaxAbsScaler,
    StandardScaler,
    SplineTransformer,
    RobustScaler,
)
from tensorflow.keras.layers import PReLU
import tensorflow as tf
from sklearn.model_selection import KFold
from tensorflow.keras.metrics import R2Score
from optuna.integration import TFKerasPruningCallback
from tensorflow.keras.optimizers import Adam
from optuna.pruners import ThresholdPruner
from sklearn.preprocessing import Normalizer

from functools import partial
from optuna.samplers import (
    TPESampler,
    QMCSampler,
    GridSampler,
    BruteForceSampler,
    RandomSampler,
)
import optuna
import logging
import sys

# Loading and preprocessing the dataset

In [ ]:
data = pd.read_json("multi-task-dataset.json")

In [ ]:
class MyMinMaxScaler(MinMaxScaler):
    def inverse_transform(self, X):
        X -= self.min_
        X /= self.scale_
        return X

def log10(x):
    return np.log10(x + 1)


def log10_inv(x):
    return (10 ** x) - 1

def get_property_scalers(props):
    scalers = {}
    for propertys in props:
        mm1 = MyMinMaxScaler()
        scalers[propertys] = Pipeline([("minmax", mm1)])
    return scalers

def get_datasets(df_init):
    df = df_init.copy()

    df["scaled_value"] = df["value"]

    props = df['Property'].unique().tolist()

    # Create final dataset for ensemble
    ps_final = get_property_scalers(props)

    for prop in props:
        # train
        cond = df[df['Property'] == prop].index
        val = df.loc[cond, ["scaled_value"]]
        df.loc[cond, ["scaled_value"]] = ps_final[prop].fit_transform(val)

    training_df,val_df=train_test_split(df,test_size=0.2,random_state=0,stratify=df['scala'])
 
    k_fold_datasets = []
    skf = StratifiedKFold(n_splits=5,shuffle=False)
    for train_index, val_index in skf.split(training_df, training_df['scala']):

        k_train = training_df.iloc[train_index].copy()
        k_val = training_df.iloc[val_index].copy()

        k_fold_datasets.append({"train": k_train, "val": k_val})

    return k_fold_datasets,val_df

total,val=get_datasets(data)

# Using optuna and tensorflow to run multi-horizon deep operator networks

In [ ]:
import optuna
from tensorflow.keras.metrics import R2Score
import tensorflow as tf
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    PReLU,
    LeakyReLU,
    Activation,
    Input,
    Dot,
    Add,
    Lambda,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import numpy as np
from pathlib import Path

train_input = np.asarray(
    list(total[i]["train"]["combined_features_polyBERT"]), dtype=float
)
train_cycles = np.asarray(list(total[i]["train"]["Cycles"]), dtype=float)
tri2 = train_cycles
train_target = np.asarray(list(total[i]["train"]["scaled_value"]), dtype=float)

test_input = np.asarray(
    list(total[i]["val"]["combined_features_polyBERT"]), dtype=float
)
tei2 = np.asarray(list(total[i]["val"]["Cycles"]), dtype=float)
test_target = np.asarray(list(total[i]["val"]["scaled_value"]), dtype=float)

seed = 123
batch_size = 50

np.random.seed(seed)
tf.random.set_seed(seed)


def params(
    trial, name, input_layer, n_layers_range=(2, 10), n_units_range=(64, 512), step=64
):
    x = input_layer
    n_layers = trial.suggest_int(f"n_layers_{name}", *n_layers_range)
    for num in range(n_layers):
        num_hidden = trial.suggest_int(
            f"n_units_{name}_l{num}", *n_units_range, step=step
        )
        x = Dense(num_hidden)(x)

        activation_name = trial.suggest_categorical(
            f"actvtn_{name}_l{num}",
            ["relu", "LeakyReLU", "selu", "gelu", "prelu", "linear"],
        )
        if activation_name == "prelu":
            x = PReLU()(x)
        elif activation_name == "LeakyReLU":
            x = LeakyReLU()(x)
        else:
            x = Activation(activation_name)(x)

        n_drop = trial.suggest_float(f"n_drop_{name}_l{num}", 0, 0.7)
        x = Dropout(n_drop)(x)
    return x


def create_model(trial):

    trunk_in = Input(shape=(1,), name="trunk")
    processing_conditions = Input(shape=(1809,), name="processing_conditions")

    # Branch Net
    branch_x = params(trial, "branch", processing_conditions)
    latent_dim = trial.suggest_int("latent_dim", 10, 100)

    act_latent = trial.suggest_categorical(
        "actvtn_latent_branch", ["relu", "LeakyReLU", "selu", "gelu", "prelu", "linear"]
    )
    branch_out = Dense(latent_dim)(branch_x)
    if act_latent == "prelu":
        branch_out = PReLU()(branch_out)
    elif act_latent == "LeakyReLU":
        branch_out = LeakyReLU()(branch_out)
    else:
        branch_out = Activation(act_latent)(branch_out)

    trunk_x = params(trial, "trunk", trunk_in)

    act_latent_trunk = trial.suggest_categorical(
        "actvtn_latent_trunk", ["relu", "LeakyReLU", "selu", "gelu", "prelu", "linear"]
    )
    trunk_out = Dense(latent_dim)(trunk_x)
    if act_latent_trunk == "prelu":
        trunk_out = PReLU()(trunk_out)
    elif act_latent_trunk == "LeakyReLU":
        trunk_out = LeakyReLU()(trunk_out)
    else:
        trunk_out = Activation(act_latent_trunk)(trunk_out)

    # Dot Product
    dot_output = Dot(axes=-1, normalize=False)([branch_out, trunk_out])

    # Bias Net (Processing Conditions)
    ps = Lambda(lambda x: x[:, -9:])(processing_conditions)

    # Make bias net fully tunable too
    bias_x = params(
        trial, "bias", ps, n_layers_range=(2, 5), n_units_range=(4, 32), step=4
    )

    # Final Bias output (1 unit)
    act_bias_out = trial.suggest_categorical(
        "actvtn_bias_out", ["relu", "LeakyReLU", "selu", "gelu", "prelu", "linear"]
    )
    bias_out = Dense(1)(bias_x)
    if act_bias_out == "prelu":
        bias_out = PReLU()(bias_out)
    elif act_bias_out == "LeakyReLU":
        bias_out = LeakyReLU()(bias_out)
    else:
        bias_out = Activation(act_bias_out)(bias_out)

    output = Add()([dot_output, bias_out])

    model = Model(inputs=[processing_conditions, trunk_in], outputs=output)
    return model

def objective(trial, study_name):
    train_dataset = tf.data.Dataset.from_tensor_slices(
        ((train_input, tri2), train_target)
    )
    test_dataset = tf.data.Dataset.from_tensor_slices(((test_input, tei2), test_target))

    train_dataset = (
        train_dataset.cache().batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
    )
    test_dataset = (
        test_dataset.cache().batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
    )

    learning_rate = trial.suggest_float("learning_rate", 5e-4, 5e-3, log=True)
    optimizer = Adam(learning_rate=learning_rate, clipnorm=1.0)

    metric = R2Score()
    model = create_model(trial)

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=[
            metric,
            tf.keras.metrics.RootMeanSquaredError(name="root_mean_squared_error"),
        ],
    )
    callbacks = [
        tf.keras.callbacks.ReduceLROnPlateau(
            factor=0.8, monitor="val_loss", verbose=True
        ),
        tf.keras.callbacks.EarlyStopping(patience=40, monitor="val_loss", verbose=True),
    ]

    fit = model.fit(
        train_dataset,
        epochs=1000,
        validation_data=test_dataset,
        batch_size=batch_size,
        verbose=2,
        callbacks=callbacks,
    )

    pt = Path(f"{study_name}/{trial.number}.keras")
    pt.parent.mkdir(parents=True, exist_ok=True)
    model.save(pt)

    return np.mean([fit.history["val_loss"][-1], fit.history["loss"][-1]])


In [ ]:
from functools import partial
import gc

from optuna.pruners import MedianPruner
from optuna.samplers import (
    TPESampler,
    QMCSampler,
    GridSampler,
    BruteForceSampler,
    RandomSampler,
)
import optuna
import logging
import sys

num = i
name_num = f"df-don-corr/{num}"
p = Path(name_num)
p.parent.mkdir(parents=True, exist_ok=True)
storage_name = "sqlite:///{}.db".format(name_num)

sampler = TPESampler(seed=42)

study = optuna.create_study(
    study_name=name_num,
    direction="minimize",
    storage=storage_name,
    load_if_exists=True,
    sampler=sampler,
    pruner=optuna.pruners.PercentilePruner(percentile=95),
)

obj = partial(objective, study_name=name_num)

study.optimize(obj, n_trials=500)